In [3]:
from pypdf import PdfReader
import pymupdf
import re
from pathlib import Path

In [ ]:
folderpath = Path("IAL D1 Papers/Past Papers") if Path("IAL D1 Papers/Past Papers").is_dir() else Path("Past Papers")
PastPapers = []
for item in sorted(folderpath.glob("*.pdf")):
    print(item.name)
    PastPapers.append(item.name)

In [ ]:
topics = {
    "Linear Programming": [
        "linear programming",
    ],
    "Critical Path Analysis": [
        "activity network",
        "precedence table",
        "cascade chart",
        "early event",
        "late event",
        "critical path",
        "total float",
        "scheduling diagram",
    ],
    "Algorithms": [
        "bin packing",
        "bubble sort",
        "quick sort",
        "binary search",
        "flow chart",
        "flowchart",
        "following algorithm",
    ],
    "Graph II": [
        "dijkstra",
        "nearest neighbour",
        "route inspection",
        "inspection route",
        "salesperson",
        "visit each",
        "at least once",
        "traversed twice",
    ],
    "Graph I": [
        "kruskal",
        "prim",
        "minimum spanning",
        "spanning tree",
    ],
}

In [ ]:
grouped_pdfs = {topic: pymupdf.open() for topic in topics}
grouped_pdfs["Unclassified"] = pymupdf.open()
counts = {topic: 0 for topic in grouped_pdfs}

all_files = sorted(folderpath.glob("*.pdf"))

for file_path in all_files:
    print(f"--- Processing: {file_path.name} ---")
    src_doc = pymupdf.open(file_path)
    total = src_doc.page_count
    
    # 1. FIND THE HARD SPLIT (The "Total for Paper" marker)
    # This page is the end of the QP. The NEXT page is the AB cover.
    split_point = total // 2
    for i in range(total):
        text = src_doc[i].get_text().lower()
        if "total for paper" in text:
            split_point = i + 1
            break
    
    # 2. MAP THE QUESTIONS (Only in the first half)
    qp_map = {}
    for p in range(split_point):
        text = src_doc[p].get_text()
        # Regex looks for "1." or "Question 1" at the very start of a line
        match = re.search(r"^\s*(\d+)\s*[\.\:]", text, re.MULTILINE)
        if match:
            q_num = int(match.group(1))
            if 1 <= q_num <= 12 and q_num not in qp_map:
                qp_map[q_num] = p

    # 3. MAP THE ANSWERS (Only in the second half)
    # We look for the first page of each answer section
    ab_map = {}
    for p in range(split_point, total):
        text = src_doc[p].get_text()
        match = re.search(r"^\s*(\d+)\s*[\.\:]", text, re.MULTILINE)
        if match:
            q_num = int(match.group(1))
            if 1 <= q_num <= 12 and q_num not in ab_map:
                ab_map[q_num] = p

    # 4. STITCH THEM TOGETHER
    sorted_qs = sorted(qp_map.keys())
    for idx, q_num in enumerate(sorted_qs):
        # QP Range
        start_q = qp_map[q_num]
        end_q = qp_map[sorted_qs[idx+1]] if idx+1 < len(sorted_qs) else split_point
        qp_text = "".join(src_doc[p].get_text() for p in range(start_q, end_q)).lower()
        target_topic = "Unclassified"
        for topic, keywords in topics.items():
            if any(keyword in qp_text for keyword in keywords):
                target_topic = topic
                break
        counts[target_topic] += 1
        print(f"  Q{q_num} -> {target_topic}")
        
        # Add Question Pages
        grouped_pdfs[target_topic].insert_pdf(src_doc, from_page=start_q, to_page=end_q-1)
        
        # Add Answer Pages (The range from Answer N to Answer N+1)
        if q_num in ab_map:
            start_a = ab_map[q_num]
            # Find the start of the NEXT answer in the AB
            next_a_vals = [v for k, v in ab_map.items() if v > start_a]
            end_a = min(next_a_vals) if next_a_vals else total
            grouped_pdfs[target_topic].insert_pdf(src_doc, from_page=start_a, to_page=end_a-1)

    src_doc.close()

print("\nQuestions per topic")
for topic, count in counts.items():
    print(f"  {topic}: {count}")

# 5. Save
output_dir = folderpath.parent / "D1_Sorted_Final"
output_dir.mkdir(exist_ok=True)
for topic, pdf_doc in grouped_pdfs.items():
    if pdf_doc.page_count > 0:
        pdf_doc.save(output_dir / f"{topic}.pdf")
    pdf_doc.close()